# 🤖 Phase 4 — Model Training

**Project:** Facial Skin Condition Detection  
**Level:** MCA Academic Project  

---

## Overview

This notebook trains **7 traditional ML classifiers** on the extracted 58-dimensional feature vectors.

### Models Trained

| # | Algorithm | Key Parameters |
|---|-----------|---------------|
| 1 | **SVM** | RBF kernel, C=10, class_weight=balanced |
| 2 | **Random Forest** | 200 trees, class_weight=balanced |
| 3 | **KNN** | k=7, distance-weighted, Euclidean |
| 4 | **Decision Tree** | max_depth=10, class_weight=balanced |
| 5 | **Logistic Regression** | C=1.0, max_iter=1000 |
| 6 | **Naive Bayes** | Gaussian |
| 7 | **XGBoost** | 200 estimators, lr=0.1, max_depth=6 |

### Pipeline
```
features.csv → 80/20 Stratified Split → StandardScaler → 
5-Fold CV for each model → Best model selected → Saved to models/
```

> **No deep learning.** All models use traditional ML on handcrafted features.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

from train_models import run_training

print('Imports OK')

## 1. Load Feature Dataset

In [ ]:
features_path = PROJECT_ROOT / 'features' / 'features.csv'

if not features_path.exists():
    print('⚠ features.csv not found. Run notebook 03 first.')
else:
    df = pd.read_csv(features_path)
    print(f'Loaded: {df.shape}')
    print(df['label'].value_counts())
    df.head(3)

## 2. Train All Models

The cell below calls `run_training()` from `src/train_models.py`, which:
- Performs stratified 80/20 split
- Scales with `StandardScaler`
- Runs 5-fold cross-validation for each of the 7 models
- Saves best model artefacts to `models/`

In [ ]:
payload = run_training()

## 3. Results Summary Table

In [ ]:
if 'payload' in dir():
    model_results = payload['model_results']
    best_name     = payload['best_model_name']

    rows = []
    for name, res in model_results.items():
        rows.append({
            'Model':        name,
            'CV Mean Acc':  round(res['cv_mean'], 4),
            'CV Std':       round(res['cv_std'],  4),
            'Test Acc':     round(res['test_accuracy'], 4),
        })

    results_df = pd.DataFrame(rows).sort_values('CV Mean Acc', ascending=False).reset_index(drop=True)
    results_df.index += 1
    print(f'Best model: {best_name}')
    results_df

## 4. CV Accuracy Comparison Plot

In [ ]:
if 'results_df' in dir():
    palette = ['#E74C3C', '#E67E22', '#F1C40F', '#2ECC71', '#3498DB', '#9B59B6', '#1ABC9C']
    fig, ax = plt.subplots(figsize=(10, 5))
    models_sorted = results_df['Model'].tolist()
    cv_accs  = results_df['CV Mean Acc'].tolist()
    cv_stds  = results_df['CV Std'].tolist()

    bars = ax.barh(models_sorted[::-1], cv_accs[::-1],
                   color=palette[:len(models_sorted)], edgecolor='white', linewidth=0.5)
    ax.errorbar(cv_accs[::-1], range(len(models_sorted)),
                xerr=cv_stds[::-1], fmt='none', color='black', capsize=4)
    ax.set_xlabel('5-Fold CV Accuracy', fontsize=12)
    ax.set_title('ML Model Comparison', fontsize=13, fontweight='bold')
    ax.set_xlim(0, 1.1)
    for bar, val in zip(bars, cv_accs[::-1]):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10)
    plt.tight_layout()
    (PROJECT_ROOT / 'reports').mkdir(exist_ok=True)
    plt.savefig(PROJECT_ROOT / 'reports' / 'training_cv_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved → reports/training_cv_comparison.png')

## 5. Learning Curve for Best Model

In [ ]:
if 'payload' in dir():
    from sklearn.model_selection import learning_curve
    import pickle

    with open(PROJECT_ROOT / 'models' / 'model_results.pkl', 'rb') as f:
        p = pickle.load(f)

    best_model = p['best_model']
    X_test     = p['X_test']
    y_test     = p['y_test']

    # Reload full scaled data
    feat_cols = [c for c in df.columns if c not in ('label', 'filepath')]
    X_raw = df[feat_cols].values.astype(np.float32)
    le    = LabelEncoder()
    y_all = le.fit_transform(df['label'].values)
    scaler = p['scaler']
    X_all  = scaler.transform(X_raw)

    curve_out = learning_curve(
        best_model, X_all, y_all,
        cv=5, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 8),
        scoring='accuracy'
    )
    train_sizes, train_scores, val_scores = curve_out[0], curve_out[1], curve_out[2]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='#3498DB', label='Train Accuracy')
    ax.fill_between(train_sizes,
                    train_scores.mean(axis=1) - train_scores.std(axis=1),
                    train_scores.mean(axis=1) + train_scores.std(axis=1),
                    alpha=0.15, color='#3498DB')
    ax.plot(train_sizes, val_scores.mean(axis=1), 'o-', color='#E74C3C', label='CV Accuracy')
    ax.fill_between(train_sizes,
                    val_scores.mean(axis=1) - val_scores.std(axis=1),
                    val_scores.mean(axis=1) + val_scores.std(axis=1),
                    alpha=0.15, color='#E74C3C')
    ax.set_xlabel('Training Set Size', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(f'Learning Curve — {p["best_model_name"]}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'reports' / 'learning_curve.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved → reports/learning_curve.png')

---
## Summary

- **7 models** trained with 5-fold stratified cross-validation
- Best model selected by CV accuracy → saved to `models/best_model.pkl`
- Scaler saved → `models/scaler.pkl`
- Full results saved → `models/model_results.pkl`

**Next:** Run `05_model_evaluation.ipynb` for confusion matrices and detailed metrics.